# Comprehensive Sales EDA — Superstore Dataset

**Layers covered:**
1. Core EDA — distributions, quality checks, YoY trend  
2. PySpark Aggregations — GROUP BY + window functions  
3. Business Intelligence Metrics — KPIs, top/bottom sub-categories  
4. Advanced Analytics — correlation, RFM-lite, discount bands, shipping  
5. Predictive Insight — Linear Regression (R², MAE, feature importance)  

In [ ]:
import sys, os
# Make sure src/ is importable when running from notebooks/
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px
import plotly.graph_objects as go

from src.data_loader import load_data, validate_data, get_data_summary
from src.eda_analysis import (
    get_kpi_metrics, get_yoy_trend, get_monthly_trend,
    get_top_products, get_category_summary, get_subcategory_margin,
    get_top_bottom_subcategories, get_region_analysis, get_segment_analysis,
    get_bi_metrics, get_discount_analysis, get_correlation_matrix,
    get_rfm_lite, get_shipping_analysis, run_regression
)
from src.spark_analysis import run_aggregations, save_results
from src.utils import PYSPARK_CODE

sns.set_theme(style='darkgrid', palette='muted')
plt.rcParams['figure.figsize'] = (12, 5)
print('Imports OK')

## Layer 1 — Core EDA

In [ ]:
df = load_data('../data/Dataset_Superstore.csv')
print(f'Shape: {df.shape}')
df.head()

In [ ]:
validation = validate_data(df)
print('=== Data Quality Report ===')
print(f"Shape            : {validation['shape']}")
print(f"Duplicate rows   : {validation['duplicate_rows']}")
print(f"Missing values   : {validation['missing_values'] or 'None'}")
print(f"Date range       : {validation['date_range'][0].date()} → {validation['date_range'][1].date()}")
validation['numeric_summary']

In [ ]:
# Distributions
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
for ax, col, color in zip(axes, ['Sales', 'Profit', 'Discount'],
                           ['#60a5fa', '#34d399', '#f59e0b']):
    ax.hist(df[col], bins=60, color=color, edgecolor='none', alpha=0.85)
    ax.set_title(f'{col} Distribution')
    ax.set_xlabel(col)
plt.tight_layout()
plt.savefig('../outputs/figures/distributions.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# YoY trend
trend = get_yoy_trend(df)
fig, ax1 = plt.subplots(figsize=(10, 4))
ax2 = ax1.twinx()
ax1.bar(trend['Year'], trend['Revenue'], color='#60a5fa', alpha=0.6, label='Revenue')
ax2.plot(trend['Year'], trend['Profit Margin %'], 'o-', color='#f59e0b', lw=2.5, label='Margin %')
ax1.set_ylabel('Revenue ($)')
ax2.set_ylabel('Profit Margin (%)')
ax1.set_title('Year-over-Year Revenue & Profit Margin')
plt.tight_layout()
plt.savefig('../outputs/figures/yoy_trend.png', dpi=150, bbox_inches='tight')
plt.show()
trend

## Layer 2 — PySpark Aggregations

In [ ]:
# Display the PySpark code used
print(PYSPARK_CODE)

In [ ]:
# Run aggregations (PySpark if available, pandas fallback otherwise)
agg_df, monthly_df, used_spark = run_aggregations(df)
engine = 'PySpark' if used_spark else 'pandas (PySpark not installed)'
print(f'Engine used: {engine}')
print(f'Aggregation rows: {len(agg_df)}')
agg_df.head(10)

In [ ]:
# Running total chart
fig, ax = plt.subplots(figsize=(14, 4))
ax.bar(monthly_df['YearMonth'], monthly_df['Monthly_Sales'], color='#60a5fa', alpha=0.6, label='Monthly Sales')
ax2 = ax.twinx()
ax2.plot(monthly_df['YearMonth'], monthly_df['Running_Total_Sales'], 'o-',
         color='#f59e0b', lw=2, label='Running Total')
ax.set_xticklabels(monthly_df['YearMonth'], rotation=90, fontsize=7)
ax.set_ylabel('Monthly Sales ($)')
ax2.set_ylabel('Running Total ($)')
ax.set_title('Monthly Sales + Running Total (PySpark Window Function)')
plt.tight_layout()
plt.savefig('../outputs/figures/running_total.png', dpi=150, bbox_inches='tight')
plt.show()

# Save results
save_results(agg_df, monthly_df, '../outputs')
print('Saved to outputs/spark_results.csv and outputs/spark_monthly.csv')

## Layer 3 — Business Intelligence Metrics

In [ ]:
kpis = get_kpi_metrics(df)
print('=== Headline KPIs ===')
for k, v in kpis.items():
    if isinstance(v, float):
        print(f'  {k:30s}: {v:,.2f}')
    else:
        print(f'  {k:30s}: {v:,}')

In [ ]:
bi = get_bi_metrics(df)
print('=== Top 3 Most Profitable Sub-Categories ===')
print(bi['top3_subcategories'].to_string(index=False))
print()
print('=== Bottom 3 Loss-Making Sub-Categories ===')
print(bi['bottom3_subcategories'].to_string(index=False))
print()
print(f"Best Region by Margin : {bi['best_region_name']} ({bi['best_region_margin']:.1f}%)")
print(f"Best Segment by AOV   : {bi['best_segment_name']} (${bi['best_segment_aov']:,.2f})")
if bi['breakeven_discount'] is not None:
    print(f"Discount Breakeven    : {bi['breakeven_discount']*100:.0f}% (profit turns negative here)")
else:
    print('Discount Breakeven    : No clear threshold found')

## Layer 4 — Advanced Analytics

In [ ]:
# Correlation heatmap
corr = get_correlation_matrix(df)
plt.figure(figsize=(7, 5))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            linewidths=0.5, square=True)
plt.title('Correlation Matrix: Sales · Profit · Discount · Quantity')
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# RFM-lite
rfm = get_rfm_lite(df)
top20 = rfm[rfm['Tier'] == 'Top 20%']
rev_share = top20['Sales'].sum() / rfm['Sales'].sum() * 100
print(f'Top 20% of customers ({len(top20):,}) generate {rev_share:.1f}% of total revenue')
rfm.head(10)

In [ ]:
# Discount impact by band
bands, breakeven, df_bands = get_discount_analysis(df)
print('=== Discount Band Analysis ===')
print(bands.to_string(index=False))
if breakeven is not None:
    print(f'\n** Profit turns negative at ~{breakeven*100:.0f}% discount **')

# Bar chart
fig, ax = plt.subplots(figsize=(8, 4))
colors = ['#22c55e' if v >= 0 else '#ef4444' for v in bands['Avg_Profit_Margin']]
ax.bar(bands['Discount Band'].astype(str), bands['Avg_Profit_Margin'], color=colors)
ax.axhline(0, color='gray', linestyle='--')
ax.set_ylabel('Avg Profit Margin (%)')
ax.set_title('Profit Margin by Discount Band')
plt.tight_layout()
plt.savefig('../outputs/figures/discount_impact.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Shipping mode analysis
ship = get_shipping_analysis(df)
print(ship.to_string(index=False))

fig, ax = plt.subplots(figsize=(8, 3))
bars = ax.barh(ship['Ship Mode'], ship['Profit Margin %'],
               color=['#22c55e' if v >= 0 else '#ef4444' for v in ship['Profit Margin %']])
ax.axvline(0, color='gray', linestyle='--')
ax.set_xlabel('Profit Margin (%)')
ax.set_title('Profit Margin by Shipping Mode')
plt.tight_layout()
plt.show()

## Layer 5 — Predictive Insight (Linear Regression)

In [ ]:
reg = run_regression(df)
print(f"R² Score : {reg['r2']:.4f}")
print(f"MAE      : ${reg['mae']:,.2f}")
print(f"Intercept: ${reg['intercept']:,.2f}")
print()
print('=== Feature Coefficients ===')
print(reg['coefficients'].to_string(index=False))
print()
print('Interpretation:')
for _, row in reg['coefficients'].iterrows():
    direction = 'increases' if row['Coefficient'] > 0 else 'decreases'
    print(f"  Each 1-unit increase in {row['Feature']} {direction} Profit by ${abs(row['Coefficient']):.4f}")

In [ ]:
# Feature importance bar chart
fig, ax = plt.subplots(figsize=(7, 3))
colors = ['#22c55e' if v >= 0 else '#ef4444' for v in reg['coefficients']['Coefficient']]
ax.barh(reg['coefficients']['Feature'], reg['coefficients']['Coefficient'], color=colors)
ax.axvline(0, color='gray', linestyle='--')
ax.set_xlabel('Coefficient (Effect on Profit $)')
ax.set_title('Linear Regression — Feature Importance')
plt.tight_layout()
plt.savefig('../outputs/figures/feature_importance.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# Example prediction
import pandas as pd
example = pd.DataFrame([[500, 0.2, 3]], columns=['Sales', 'Discount', 'Quantity'])
pred = reg['model'].predict(example)[0]
print(f'Example order — Sales: $500, Discount: 20%, Quantity: 3')
print(f'Predicted Profit: ${pred:,.2f}')